In [1]:
# ---- cell 1: imports (your existing cell, plus importlib) ----
from google.colab import drive
drive.mount('/content/drive')
!pip install -q wfdb
import os, sys, io, contextlib, hashlib, inspect, importlib
os.chdir('/content/drive/MyDrive/FYP'); sys.path.insert(0, os.getcwd())

import numpy as np, pandas as pd, torch
from scipy.special import softmax
from src import config as C
from src.data import load_beats
from src.segment import build_dataset
from src.model import BeatCNN, ResBeatCNN
from src.train2 import fit, get_logits, DEVICE
import src.adoption as A

torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False
print('device:', DEVICE)

Mounted at /content/drive
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 163.9/163.9 kB 4.8 MB/s eta 0:00:00
device: cuda


In [2]:


# ---- cell 2: data + FIXED folds (201 and 202 are the same subject -> same fold) ----
df = load_beats()
d1 = df[df.split == 'DS1']
X, RR, y, rec, ridx = build_dataset(d1)

FOLDS = {
    0: [109, 118, 119, 115, 230],
    1: [207, 106, 112, 122],
    2: [208, 124, 114, 116],
    3: [203, 209, 101, 108],
    4: [201, 202, 205, 215, 220, 223],
}
rec2fold = {r: k for k, rs in FOLDS.items() for r in rs}
assert sorted(rec2fold) == sorted(set(int(r) for r in rec))
assert rec2fold[201] == rec2fold[202]
fold_id = np.array([rec2fold[int(r)] for r in rec])
for k in range(5):
    print(f'fold {k}: val beats N/S/V/F', np.bincount(y[fold_id == k], minlength=4))


fold 0: val beats N/S/V/F [10405    96   499     2]
fold 1: val beats N/S/V/F [8062  109  730    0]
fold 2: val beats N/S/V/F [7242   46 1191  381]
fold 3: val beats N/S/V/F [8748  392  462    3]
fold 4: val beats N/S/V/F [13460   356   925    29]


In [3]:

# ---- cell 3: CV runner; the recipe is IDENTICAL for both models ----
SEEDS = [42, 43, 44]
RECIPE = dict(epochs=10, aug=True, wd=1e-3)       # your current baseline recipe

def run_cv(model_fn, seeds=SEEDS, **kw):
    outs = {}
    for s in seeds:
        o = np.zeros((len(X), 4), np.float32)
        for k in range(5):
            va = np.where(fold_id == k)[0]; tr = np.where(fold_id != k)[0]
            with contextlib.redirect_stdout(io.StringIO()):
                m = fit(X[tr], RR[tr], y[tr], seed=s, model_fn=model_fn, **kw)
            o[va] = get_logits(m, X[va], RR[va])
        outs[s] = o
        print('seed', s, 'done')
    return outs


In [6]:
print()

# ---- cell 4: BASELINE on the fixed folds, then read its seed std ----
oof_base = run_cv(BeatCNN, **RECIPE)
np.savez(C.PROC / 'oof_ds1_v6_base.npz', logits=np.stack([oof_base[s] for s in SEEDS]),
         y=y, rec=rec, ridx=ridx, fold=fold_id, seeds=SEEDS)
f1s = [A.metrics(y, oof_base[s])['macro_f1'] for s in SEEDS]
print('baseline macro-F1(NSV) per seed', np.round(f1s, 4), '| std', round(float(np.std(f1s, ddof=1)), 4))
print('suggested MIN_GAIN ~', round(max(0.01, 2 * float(np.std(f1s, ddof=1))), 3))



seed 42 done
seed 43 done
seed 44 done
baseline macro-F1(NSV) per seed [0.5122 0.5736 0.4882] | std 0.044
suggested MIN_GAIN ~ 0.088


In [7]:

# ---- cell 5: lock the rule, then run the residual model ----
importlib.reload(A)
print(A.RULE)
print('adoption.py sha256:', hashlib.sha256(inspect.getsource(A).encode()).hexdigest()[:12])
assert A.RULE['MIN_GAIN'] is not None
print('params BeatCNN', sum(p.numel() for p in BeatCNN().parameters()),
      '| ResBeatCNN', sum(p.numel() for p in ResBeatCNN().parameters()))

oof_res = run_cv(ResBeatCNN, **RECIPE)
np.savez(C.PROC / 'oof_ds1_v6_res.npz', logits=np.stack([oof_res[s] for s in SEEDS]),
         y=y, rec=rec, ridx=ridx, fold=fold_id, seeds=SEEDS)


{'MIN_GAIN': 0.03, 'MIN_SEEDS_WON': 2, 'MAX_S_DROP': 0.01, 'MAX_V_DROP': 0.04, 'MAX_ECE_INCREASE': 0.025}
adoption.py sha256: 786feef2a274
params BeatCNN 26004 | ResBeatCNN 71028
seed 42 done
seed 43 done
seed 44 done


In [10]:
importlib.reload(A)
# ---- cell 6: decision + informational ensemble comparison ----
adopt = A.decide(y, oof_base, oof_res)
ens = lambda o: np.mean([o[s] for s in SEEDS], 0)
print('ensemble (info only): base', {k: round(v, 3) for k, v in A.metrics(y, ens(oof_base)).items()},
      '\n                      res ', {k: round(v, 3) for k, v in A.metrics(y, ens(oof_res)).items()})

baseline macro-F1(NSV) per seed [0.5122 0.5736 0.4882] (std 0.0440)
residual macro-F1(NSV) per seed [0.4594 0.5007 0.5018]
gain -0.0374 | wins 1/3 | dS -0.0353 | dV -0.0366 | dECE +0.0092 | F1_F base 0.002 res 0.004 (descriptive)
  [FAIL] mean macro-F1(NSV) gain >= 0.03
  [FAIL] wins on >= 2 seeds
  [FAIL] S-F1 drop <= 0.01
  [PASS] V-F1 drop <= 0.04
  [PASS] ECE(after TS) increase <= 0.025
DECISION: KEEP baseline CNN -> freeze classifier
ensemble (info only): base {'macro_f1': 0.528, 'f1_S': 0.157, 'f1_V': 0.52, 'f1_F': 0.002, 'ece_ts': 0.032, 'T': 1.289} 
                      res  {'macro_f1': 0.502, 'f1_S': 0.133, 'f1_V': 0.505, 'f1_F': 0.003, 'ece_ts': 0.044, 'T': 1.798}


In [14]:
import os; os.environ['CUBLAS_WORKSPACE_CONFIG'] = ':4096:8'
torch.use_deterministic_algorithms(True, warn_only=True)

In [15]:
d = np.load(C.PROC / 'oof_ds1_v6_base.npz')
print([round(A.metrics(d['y'], d['logits'][i])['macro_f1'], 4) for i in range(3)])  # expect 0.5122, 0.5736, 0.4882

[0.5122, 0.5736, 0.4882]


In [11]:
## SAMPLE CHECK EXPERIMENT
## TO SET THE BASE LINE
# NO NEED TO RUN THIS
from sklearn.metrics import f1_score
rows = []
for s in SEEDS:
    p = oof_base[s].argmax(1)
    f = f1_score(y, p, labels=range(4), average=None, zero_division=0)
    m = A.metrics(y, oof_base[s])
    rows.append([s, *f.round(3), round(m['macro_f1'], 4), round(m['ece_ts'], 4), round(m['T'], 2)])
tab = pd.DataFrame(rows, columns=['seed','F1_N','F1_S','F1_V','F1_F','macroNSV','ECE_TS','T'])
print(tab.to_string(index=False))
print(tab[['F1_S','F1_V','macroNSV','ECE_TS']].std(ddof=1).round(4))

# per-fold S and V F1 for seed 42 (shows which fold drags S down)
p = oof_base[42].argmax(1)
for k in range(5):
    m_ = fold_id == k
    f = f1_score(y[m_], p[m_], labels=range(4), average=None, zero_division=0)
    print(f'fold {k}: F1 N/S/V', f[:3].round(3))

 seed  F1_N  F1_S  F1_V  F1_F  macroNSV  ECE_TS    T
   42 0.866 0.103 0.568 0.002    0.5122  0.0321 1.24
   43 0.926 0.254 0.541 0.002    0.5736  0.0547 1.27
   44 0.884 0.085 0.495 0.003    0.4882  0.0492 1.63
F1_S        0.0928
F1_V        0.0369
macroNSV    0.0440
ECE_TS      0.0118
dtype: float64
fold 0: F1 N/S/V [0.775 0.    0.332]
fold 1: F1 N/S/V [0.88  0.    0.434]
fold 2: F1 N/S/V [0.925 0.003 0.925]
fold 3: F1 N/S/V [0.832 0.328 0.443]
fold 4: F1 N/S/V [0.909 0.129 0.799]


In [12]:
from sklearn.metrics import precision_recall_fscore_support as prfs
p = oof_base[42].argmax(1)
print(prfs(y, p, labels=[0,1,2], zero_division=0)[:2])   # precision, recall per class

(array([0.97369929, 0.06262492, 0.41739775]), array([0.78034935, 0.28228228, 0.88731284]))
